In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [3]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


In [4]:
model_name = 'cemh'

## Load and Filter Dataset

In [5]:
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft')
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [6]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)

## Fit and evaluate models

In [7]:
np.random.seed(1)
all_results = []
for i in tqdm(range(pdl.max_samples)):
    train_df, test_df = pdl.get_sample_split_dataset(i)
    model_output = train_df.groupby(['treatment', 'game', 'feedback_setting', 'price_rule', 'round'])[['allocative_efficiency_round']].first().groupby(['feedback_setting', 'price_rule', 'round'])['allocative_efficiency_round'].median()
    for rd in rounds:
        for n_deal_price in n_deal_prices:
            key = (rd, n_deal_price)
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query)
            test_model_df = pd.merge(model_output.reset_index().rename(columns={'allocative_efficiency_round' : 'allocative_efficiency_round_pred'}), sub_test_set, on=['feedback_setting', 'price_rule', 'round'], how='inner')
            assert test_model_df.shape[0] == sub_test_set.shape[0]
            result_test_df = sub_test_set[key_columns].copy()
            
            # Since allocative efficiency can be 0, calculate median APE, with some changes in the denominator.
            denom = test_model_df['allocative_efficiency_round'].copy()
            denom[denom==0]= test_model_df['allocative_efficiency_round_pred'].loc[denom==0]
            denom[denom==0] = 1
            
            result_test_df.loc[:, 'ae_ape'] =  np.abs(test_model_df['allocative_efficiency_round_pred'].values - test_model_df['allocative_efficiency_round'].values)/denom.values
            result_test_df.loc[:, 'sample_id'] = i
            assert not result_test_df['ae_ape'].isna().any()
            all_results.append(result_test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine Results

In [8]:
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name

## Persist Performance Results

In [10]:
all_results_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'.ft')